In [13]:
#!pip install sympy

In [14]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from Sympy_OPF_LALM_class import extract_qhd_solution_vector
from Sympy_OPF_LALM_class import initialize_qhd_acopf_log
from Sympy_OPF_LALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json

In [15]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # Convert "k1", "k2", ... into integer keys 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens


In [16]:
# Initialize the model.
# model = SympyACOPFModel()   # Enable the proximal option if needed later.
n = 5 # Select the test system size: 2, 3, or larger.

if n == 2:
    # 2-bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3-bus model
    model = SympyACOPFModel()

else:
    # n-bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom.json")
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 5 buses 6 lines and 4 generators.


In [ ]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 50
alpha = 0.3   # 对偶步长尽量小一点 simi 0.5-1.0 (2 bus)
max_outer = 600
tol = 1e-4
option = 2 # 1: QHD, 2: Gurobi
qhd_solver = "simbi"  # simbi / openjij / gurobi

# ========= 新增：初始化日志文件 =========
# alpha 调度选项
# alpha_option: "fixed" / "decay"
alpha_option = "decay"
fixed_alpha = alpha

# 若 alpha_option == "decay"，则有两种输入方式：
# 1. alpha0 + alpha_decay_factor -> alpha_(k+1) = alpha_k * alpha_decay_factor
# 2. alpha0 + alphaN -> alpha_decay_factor = (alphaN / alpha0)**(1 / max_outer)
# 若用户什么都没输入，则默认采用方案 2：alpha0=1, alphaN=1e-4
alpha0 = 1#None
alpha_decay_factor = None
alphaN = 1e-6 #None

if alpha_option == "fixed":
    if fixed_alpha <= 0:
        raise ValueError("fixed_alpha must be positive.")
    alpha = fixed_alpha
elif alpha_option == "decay":
    alpha0_use = 1.0 if alpha0 is None else alpha0
    if alpha0_use <= 0:
        raise ValueError("alpha0 must be positive.")
    if alpha_decay_factor is not None:
        if alpha_decay_factor <= 0:
            raise ValueError("alpha_decay_factor must be positive.")
        decay_factor = alpha_decay_factor
    else:
        alphaN_use = 1e-4 if alphaN is None else alphaN
        if alphaN_use <= 0:
            raise ValueError("alphaN must be positive.")
        decay_factor = (alphaN_use / alpha0_use) ** (1.0 / max_outer)
    alpha = alpha0_use
else:
    raise ValueError("alpha_option must be 'fixed' or 'decay'.")

log_file = initialize_qhd_acopf_log(model, folder="logs", option=option, qhd_solver=qhd_solver)
print("Log file:", log_file)
print(f"Alpha option: {alpha_option}")
if alpha_option == "fixed":
    print(f"Fixed alpha: {fixed_alpha}")
else:
    print(f"Decay alpha: alpha0={alpha0_use}, decay_factor={decay_factor}")

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    if alpha_option == "fixed":
        alpha = fixed_alpha
    else:
        alpha = alpha0_use * (decay_factor ** k)

    print(f"\n--- Outer Iteration {k} ---")
    print(f"alpha = {alpha:.6e}")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=0
        )
    
    bad_bounds = []
    for i, (var, bnd) in enumerate(zip(variable_list, Var_bound_list)):
        lb, ub = float(bnd[0]), float(bnd[1])
        if ub < lb:
            bad_bounds.append((i, str(var), lb, ub))

    if bad_bounds:
        print("\n=== Invalid bounds found ===")
        for item in bad_bounds:
            print(item)
        raise ValueError("Var_bound_list contains invalid bounds (ub < lb).")

    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=16,
                agents=1024,
                max_steps=5000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                #ballistic=True,
                #heated=True,
                best_only=False,
                verbose=True
            )
        elif qhd_solver == "openjij":
            qhd_model.openjij_setup(
                resolution=6,
                shots=2048,
                sampler_name="SQASampler",
                post_processing_method='TNC',
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 10000,
                    "reinitialize_state": True,
                },
            )
        elif qhd_solver == "gurobi":
            qhd_model.gurobi_setup(
                resolution=4,
                shots=20,
                embedding_scheme="unary",
                solver_mode="ising",
                time_limit=30,
                threads=0,
                log_to_console=False,
                post_processing_method='TNC',
            )
        else:
            raise ValueError(f"Unsupported qhd_solver={qhd_solver!r}. Use 'simbi', 'openjij', or 'gurobi'.")
        response = qhd_model.optimize(refine=True, verbose=0)
        #response = qhd_model.optimize(refine=False, verbose=0)

        x_new = extract_qhd_solution_vector(response, expected_len=len(variable_list))

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=alpha,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    # Keep rho fixed for stability diagnostics.
    print("Keeping rho fixed at", rho)

    # ======================================
    # (6) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(x_new)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (7) 记录日志（每轮追加）
    # ======================================
    subs_dict = {var: val for var, val in zip(model.variable_list, x_new)}
    #objective_value = float(model.objective.evalf(subs=subs_dict))

    log_file = PrintQHDACOPFResults(
        model,
        x_new,
        log_file=log_file,
        iteration=k,
        folder="logs",
        print_to_console=True,
        rho=rho,
        alpha=alpha,
        h_x=h_val,
        lambda_vec=lambda_new,
        objective_value=0,
        feasibility=check_flag,
    )

    # 如果你还想同时在屏幕上打印表格，可以再保留这一句
    # PrintQHDACOPFResults(model, x_new, iteration=k, log_file=log_file, print_to_screen=True)


    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (9) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")
print("Final log file:", log_file)

Log file: logs\Buses-5_04-28-2026_00-22-12.txt
Alpha option: decay
Decay alpha: alpha0=20, decay_factor=0.9723701463076586

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---
alpha = 2.000000e+01


Bifurcated agents:   0%|          | 0/1024 [00:06<?, ?it/s]


Minimizer: [0.5625 0.375  0.875  0.125  0.3125 0.8125 0.     0.8125 0.6875 0.625
 0.625  0.625  0.6875 0.5625 0.5625 0.5625 0.5625 0.5625 0.8125 0.25
 0.125  0.0625 0.5625 0.375  0.1875 1.     0.875  0.3125 0.75   0.5625
 0.5625 0.3125 0.1875 0.3125 0.375  0.3125 0.375  0.75   0.0625 0.4375
 0.5    0.6875 0.5625 0.625  0.75   0.25   0.75   0.0625 0.125  0.25
 0.0625 0.125  0.25   0.0625 0.1875 0.375  0.125  0.25   0.3125]
||h(x)|| = 11.12102819036627
[rho-check] ||h_old||=2.349e+00, ||h_new||=1.112e+01, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-28 00:23:32
Iteration: 0
rho: 50
alpha: 20
objective_value: 0
feasible: False
max_abs_h: 5.141918551608e+00
l2_norm_h: 1.112102819037e+01
lambda_inf_norm: 1.028383710322e+02
lambda_l2_norm: 2.224205638073e+02
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.018124	0.009630	0.990717	3.64130

Bifurcated agents:   0%|          | 0/1024 [00:05<?, ?it/s]


Minimizer: [0.5    0.375  0.375  0.25   0.75   0.     0.0625 0.875  0.6875 0.625
 0.5625 0.625  0.6875 0.4375 0.4375 0.4375 0.4375 0.4375 0.6875 0.875
 0.375  0.5    0.3125 0.625  0.375  0.5625 0.375  0.625  0.375  0.375
 0.5625 0.5    0.625  0.375  0.8125 0.6875 0.3125 1.     0.3125 0.1875
 0.5    0.875  0.0625 0.125  0.625  0.375  0.6875 0.375  0.3125 0.25
 0.0625 0.25   0.125  0.4375 0.5625 0.625  0.125  0.1875 0.1875]
||h(x)|| = 3.8540691528253634
[rho-check] ||h_old||=1.112e+01, ||h_new||=3.854e+00, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-28 00:24:45
Iteration: 1
rho: 50
alpha: 19.447403
objective_value: 0
feasible: False
max_abs_h: 1.955474292525e+00
l2_norm_h: 3.854069152825e+00
lambda_inf_norm: 6.538064554812e+01
lambda_l2_norm: 1.546201741258e+02
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.979826	-0.004448	0.95636

...

Minimizer: [0.1875 0.6875 0.8125 0.5625 0.6875 0.3125 0.25   0.5625 0.875  0.75
 0.6875 0.8125 0.875  0.5625 0.5625 0.5625 0.5625 0.5625 0.6875 0.0625
 0.625  1.     0.1875 0.625  0.4375 0.6875 0.375  0.75   0.125  0.75
 0.5625 0.625  0.75   0.4375 0.9375 0.8125 0.1875 0.625  0.125  0.75
 0.5    1.     0.4375 0.     0.875  0.25   0.75   0.4375 0.3125 0.25
 0.375  0.625  0.4375 0.875  0.     0.5625 0.625  0.125  0.5625]
||h(x)|| = 2.9238018441194478
[rho-check] ||h_old||=1.770e+00, ||h_new||=2.924e+00, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-28 13:02:19
Iteration: 597
rho: 50
alpha: 1.08769e-06
objective_value: 0
feasible: False
max_abs_h: 1.663271841496e+00
l2_norm_h: 2.923801844119e+00
lambda_inf_norm: 4.002548709781e+01
lambda_l2_norm: 9.757970279217e+01
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.994655	-0.009682	1.0612

Bifurcated agents:   0%|          | 0/1024 [00:05<?, ?it/s]


Minimizer: [0.     0.1875 0.5    0.8125 0.875  0.1875 0.0625 0.1875 0.875  0.875
 0.875  0.875  0.875  0.5625 0.5625 0.5625 0.5625 0.5625 0.3125 0.
 0.1875 0.5625 0.5    0.625  0.3125 0.375  0.5625 0.5    0.     0.5
 0.5    0.75   0.375  0.3125 0.8125 0.6875 0.375  0.6875 0.375  0.75
 0.5625 0.75   0.1875 0.5625 0.625  0.5    0.375  0.1875 0.1875 0.
 0.     0.0625 0.5625 0.375  0.375  0.     0.3125 0.1875 0.375 ]
||h(x)|| = 2.59250668345863
[rho-check] ||h_old||=2.924e+00, ||h_new||=2.593e+00, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-28 13:03:38
Iteration: 598
rho: 50
alpha: 1.0576373e-06
objective_value: 0
feasible: False
max_abs_h: 1.347343886586e+00
l2_norm_h: 2.592506683459e+00
lambda_inf_norm: 4.002548707484e+01
lambda_l2_norm: 9.757970291884e+01
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.029097	0.005147	1.058095	3.31

Bifurcated agents:   0%|          | 0/1024 [00:05<?, ?it/s]


Minimizer: [0.     0.4375 0.5625 0.4375 0.75   0.5625 0.6875 0.3125 0.8125 0.6875
 0.6875 0.75   0.8125 0.4375 0.4375 0.4375 0.4375 0.4375 0.25   0.8125
 0.5    0.8125 0.1875 0.375  0.375  0.5625 0.5625 0.5    0.5    0.5
 0.6875 0.625  0.5625 0.5625 0.375  0.75   0.3125 0.6875 0.1875 0.625
 0.3125 0.75   0.3125 0.25   0.75   0.375  1.     0.125  0.0625 0.1875
 0.3125 0.     0.0625 0.25   0.25   0.3125 0.1875 0.     0.6875]
||h(x)|| = 1.8215523333028292
[rho-check] ||h_old||=2.593e+00, ||h_new||=1.822e+00, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-28 13:05:06
Iteration: 599
rho: 50
alpha: 1.028415e-06
objective_value: 0
feasible: False
max_abs_h: 7.448353895467e-01
l2_norm_h: 1.821552333303e+00
lambda_inf_norm: 4.002548630884e+01
lambda_l2_norm: 9.757970138738e+01
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.011817	-0.010180	1

In [18]:
len(list(Lag.free_symbols)),list(Lag.free_symbols)

(59,
 [P_ij9,
  Q_G1,
  S_tot_sq5,
  P_ij6,
  V_sq3,
  P_ij1,
  P_ij10,
  Q_ij1,
  V_R4,
  Q_ij0,
  V_I2,
  V_I3,
  V_sq1,
  S_tot_sq8,
  V_sq0,
  P_ij3,
  S_tot_sq1,
  S_tot_sq7,
  S_tot_sq6,
  S_tot_sq4,
  V_R3,
  V_I0,
  Q_ij4,
  P_G3,
  S_tot_sq0,
  Q_ij9,
  S_tot_sq10,
  V_R2,
  Q_ij10,
  Q_ij5,
  V_R0,
  V_sq2,
  P_ij5,
  P_G1,
  S_tot_sq2,
  V_I4,
  S_tot_sq11,
  P_G0,
  Q_ij6,
  P_ij2,
  Q_ij7,
  V_sq4,
  S_tot_sq9,
  V_I1,
  Q_G0,
  P_ij7,
  P_G2,
  P_ij4,
  P_ij0,
  P_ij8,
  S_tot_sq3,
  Q_ij8,
  Q_G3,
  P_ij11,
  Q_G2,
  Q_ij11,
  Q_ij2,
  V_R1,
  Q_ij3])

In [19]:
qhd_model.SymPy.__sizeof__()

48

In [20]:
backend = qhd_model.qhd_base.backend

h = getattr(backend, "h", None)
J = getattr(backend, "J", None)


print("h:", h)
print("J:", J)
print("h shape:", len(h) if h is not None else None)   


h: [-147713.83643393376, -35.409174864754505, -32.96776861475428, -30.526362364754505, -28.084956114754277, -25.643549864754505, -23.202143614754277, -20.76073736475439, -18.31933111475439, -15.877924864754391, -13.436518614754334, -10.995112364754391, -8.553706114754391, -6.1122998647543625, -3.6708936147543767, 147674.75636545423, -147682.61177170978, -5.7470126407864655, -4.868106390786352, -3.9892001407864086, -3.1102938907863518, -2.2313876407864086, -1.3524813907863518, -0.47357514078635177, 0.4053311092135914, 1.2842373592136198, 2.16314360921362, 3.0420498592136482, 3.9209561092136056, 4.799862359213627, 5.678768609213623, 147682.5435276782, -147678.95331234855, -2.5768345295368817, -2.1862095295368817, -1.7955845295368817, -1.404959529536825, -1.0143345295368817, -0.6237095295368817, -0.23308452953688175, 0.15754047046311825, 0.5481654704631183, 0.9387904704631467, 1.3294154704631183, 1.7200404704631183, 2.1106654704631325, 2.5012904704631254, 147678.87776828944, -147680.85167